# Chapter 2 — End-to-End Machine Learning Project

This notebook follows Chapter 2 of *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow (3rd Edition)* by Aurélien Géron.

## Goal
To work through the California housing project step by step in a structured way, including:
- getting the data
- exploring the data
- creating a test set
- preparing the data
- training models
- fine-tuning models
- evaluating results

## Environment
- VS Code
- Jupyter Notebook
- Python virtual environment / selected kernel

## Notes
This notebook is being built one logical block at a time for learning and practice.

In [9]:
import os
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit

In [10]:
BOOK_ROOT = Path(r"C:\Users\dubey\Documents\UC_GitLab\Hands_On_Machine_Learning_Geron_3rd")
HOUSING_PATH = BOOK_ROOT / "datasets" / "housing"

# corrected dataset source
HOUSING_URL = "https://github.com/ageron/data/raw/main/housing.tgz"

print("Book root   :", BOOK_ROOT)
print("Housing path:", HOUSING_PATH)
print("Housing URL :", HOUSING_URL)

Book root   : C:\Users\dubey\Documents\UC_GitLab\Hands_On_Machine_Learning_Geron_3rd
Housing path: C:\Users\dubey\Documents\UC_GitLab\Hands_On_Machine_Learning_Geron_3rd\datasets\housing
Housing URL : https://github.com/ageron/data/raw/main/housing.tgz


## Block 4 — Define Helper Functions

In this block, we define two reusable functions:

- `fetch_housing_data()` to download and extract the housing dataset
- `load_housing_data()` to load the extracted CSV into a pandas DataFrame

### Why this matters
Defining reusable functions keeps the notebook clean, organized, and easy to rerun.

In [11]:
def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    housing_path.mkdir(parents=True, exist_ok=True)
    tgz_path = housing_path / "housing.tgz"
    urllib.request.urlretrieve(housing_url, tgz_path)

    with tarfile.open(tgz_path) as housing_tgz:
        housing_tgz.extractall(path=housing_path)


def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = housing_path / "housing.csv"
    return pd.read_csv(csv_path)

## Block 5 — Download and Load the Housing Dataset

In this block, we will:
- download the California housing dataset
- extract the compressed file
- load the CSV into a pandas DataFrame
- confirm that the dataset is available in memory

### Why this matters
This is the first step where we move from setup code to actual project data.
Once the dataset is loaded, we can begin inspecting its structure and preparing it for the machine learning workflow.

In [12]:
fetch_housing_data()
housing = load_housing_data()

print("Dataset loaded successfully.")
print("Shape of housing DataFrame:", housing.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\dubey\\Documents\\UC_GitLab\\Hands_On_Machine_Learning_Geron_3rd\\datasets\\housing\\housing.csv'

## Block 6 — Inspect the Extracted Housing Files

The dataset archive was downloaded, but the CSV was not found at the expected path.
In this block, we inspect the extracted folder structure to locate the actual `housing.csv` file.

In [13]:
for path in HOUSING_PATH.rglob("*"):
    print(path)

C:\Users\dubey\Documents\UC_GitLab\Hands_On_Machine_Learning_Geron_3rd\datasets\housing\housing
C:\Users\dubey\Documents\UC_GitLab\Hands_On_Machine_Learning_Geron_3rd\datasets\housing\housing.tgz
C:\Users\dubey\Documents\UC_GitLab\Hands_On_Machine_Learning_Geron_3rd\datasets\housing\housing\housing.csv


## Block 7 — Fix the CSV Load Path

The extracted archive placed `housing.csv` inside a nested `housing` folder.

In this block, we update the loader function so it reads the CSV from the correct location.

In [14]:
def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = housing_path / "housing" / "housing.csv"
    return pd.read_csv(csv_path)

## Block 8 — Load the Housing Dataset

Now that the CSV path is corrected, we load the housing dataset into a pandas DataFrame and confirm its shape.

In [15]:
housing = load_housing_data()

print("Dataset loaded successfully.")
print("Shape of housing DataFrame:", housing.shape)

Dataset loaded successfully.
Shape of housing DataFrame: (20640, 10)


## Block 9 — Inspect the First Few Rows

Before creating a test set, we quickly inspect the dataset to understand what the columns look like.
This helps us see the target column and the feature structure.

In [16]:
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Block 10 — Inspect the Data Structure

In this block, we inspect the structure of the housing dataset before creating the test set.

We check:
- column names
- data types
- number of non-null values
- a statistical summary of numeric columns

### Why this matters
This helps us understand the dataset we are about to split and confirms that `median_income` is available for the stratified sampling step used in the book.

In [17]:
print("=== DataFrame Info ===")
housing.info()

print("\n=== Summary Statistics ===")
display(housing.describe())

=== DataFrame Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB

=== Summary Statistics ===


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


## Block 11 — Create a Simple Random Train/Test Split

In this block, we create a basic random split of the dataset:
- 80% training set
- 20% test set

This is the simplest test-set creation approach.
We will later compare it with the stratified split from the book.

In [18]:
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

print("Train set shape:", train_set.shape)
print("Test set shape :", test_set.shape)

print("\nFirst 3 rows of random test set:")
display(test_set.head(3))

Train set shape: (16512, 10)
Test set shape : (4128, 10)

First 3 rows of random test set:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
20046,-122.38,40.67,10.0,2281.0,444.0,1274.0,438.0,2.2120,65600.0,INLAND
3024,-118.37,33.83,35.0,1207.0,207.0,601.0,213.0,4.7308,353400.0,<1H OCEAN
15663,-117.24,32.72,39.0,3089.0,431.0,1175.0,432.0,7.5925,466700.0,NEAR OCEAN


## Block 12 — Create Income Categories for Stratified Sampling

The book uses `median_income` to create income categories.
These categories help ensure that the train and test sets preserve the overall income distribution of the dataset.

### Why this matters
A pure random split can accidentally distort important subgroup proportions.
Stratified sampling helps make the test set more representative.

In [19]:
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

print("Income category counts:")
display(housing["income_cat"].value_counts().sort_index())

print("\nIncome category proportions:")
display(housing["income_cat"].value_counts(normalize=True).sort_index())

Income category counts:


income_cat
1     822
2    6581
3    7236
4    3639
5    2362
Name: count, dtype: int64


Income category proportions:


income_cat
1    0.039826
2    0.318847
3    0.350581
4    0.176308
5    0.114438
Name: proportion, dtype: float64

## Block 13 — Create a Stratified Train/Test Split

In this block, we create a stratified split based on `income_cat`.

### Why this matters
This preserves the distribution of income categories in both the training set and the test set, making the split more representative than a purely random split.

In [20]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index].copy()
    strat_test_set = housing.loc[test_index].copy()

print("Stratified train set shape:", strat_train_set.shape)
print("Stratified test set shape :", strat_test_set.shape)

print("\nFirst 3 rows of stratified test set:")
display(strat_test_set.head(3))

Stratified train set shape: (16512, 11)
Stratified test set shape : (4128, 11)

First 3 rows of stratified test set:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_cat
3905,-121.95,37.11,21.0,2387.0,357.0,913.0,341.0,7.7360,397700.0,<1H OCEAN,5
16821,-118.01,33.89,36.0,1589.0,265.0,804.0,272.0,4.6354,202900.0,<1H OCEAN,4
2900,-118.18,33.74,30.0,5915.0,1750.0,2136.0,1503.0,4.0968,310000.0,NEAR OCEAN,3


## Block 14 — Compare Random vs Stratified Income Category Proportions

In this block, we compare:
- the overall dataset income proportions
- the random test set income proportions
- the stratified test set income proportions

This shows why stratified sampling is preferred here.

In [21]:
def income_cat_proportions(data):
    return data["income_cat"].value_counts(normalize=True).sort_index()

compare_props = pd.DataFrame({
    "Overall": income_cat_proportions(housing),
    "Random Test": income_cat_proportions(test_set),
    "Stratified Test": income_cat_proportions(strat_test_set),
}).sort_index()

compare_props["Rand. % Error"] = 100 * compare_props["Random Test"] / compare_props["Overall"] - 100
compare_props["Strat. % Error"] = 100 * compare_props["Stratified Test"] / compare_props["Overall"] - 100

display(compare_props)

KeyError: 'income_cat'

## Fix Before Block 14 — Recreate the Random Split with `income_cat`

The earlier random test set was created before the `income_cat` helper column existed.
To compare random and stratified income-category proportions fairly, we recreate the random split from the updated dataset.

In [22]:
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

print("Random split recreated successfully.")
print("Train set shape:", train_set.shape)
print("Test set shape :", test_set.shape)
print("\nDoes random test set contain income_cat?", "income_cat" in test_set.columns)

Random split recreated successfully.
Train set shape: (16512, 11)
Test set shape : (4128, 11)

Does random test set contain income_cat? True


## Block 14 — Compare Random vs Stratified Income Category Proportions

In this block, we compare:
- the overall dataset income proportions
- the random test set income proportions
- the stratified test set income proportions

This shows why stratified sampling is preferred here.

In [23]:
def income_cat_proportions(data):
    return data["income_cat"].value_counts(normalize=True).sort_index()

compare_props = pd.DataFrame({
    "Overall": income_cat_proportions(housing),
    "Random Test": income_cat_proportions(test_set),
    "Stratified Test": income_cat_proportions(strat_test_set),
}).sort_index()

compare_props["Rand. % Error"] = 100 * compare_props["Random Test"] / compare_props["Overall"] - 100
compare_props["Strat. % Error"] = 100 * compare_props["Stratified Test"] / compare_props["Overall"] - 100

display(compare_props)

,Overall,Random Test,Stratified Test,Rand. % Error,Strat. % Error
income_cat,,,,,
1,0.039826,0.042393,0.039971,6.447689,0.364964
2,0.318847,0.307413,0.318798,-3.586081,-0.015195
3,0.350581,0.345203,0.350533,-1.533997,-0.013820
4,0.176308,0.184109,0.176357,4.424292,0.027480
5,0.114438,0.120882,0.114341,5.630821,-0.084674


## Block 15 — Remove the Helper `income_cat` Column

The `income_cat` column was created only to support stratified sampling.

Now that the train and test sets have been created and compared, we remove this helper column so the datasets return to their original structure.

In [24]:
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

print("Columns in stratified training set:")
print(strat_train_set.columns.tolist())

print("\nColumns in stratified test set:")
print(strat_test_set.columns.tolist())

print("\nShapes after dropping income_cat:")
print("Stratified train set:", strat_train_set.shape)
print("Stratified test set :", strat_test_set.shape)

Columns in stratified training set:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

Columns in stratified test set:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

Shapes after dropping income_cat:
Stratified train set: (16512, 10)
Stratified test set : (4128, 10)


## Block 16 — Final Cleanup After Test Set Creation

The `income_cat` column was only needed as a temporary helper for stratified sampling.

In this block, we remove it from:
- the full housing dataset
- the random training set
- the random test set

This keeps all datasets aligned with the original column structure.

In [25]:
for set_ in (housing, train_set, test_set):
    if "income_cat" in set_.columns:
        set_.drop("income_cat", axis=1, inplace=True)

print("income_cat removed from housing, train_set, and test_set (if present).")

print("\nColumns in housing:")
print(housing.columns.tolist())

print("\nColumns in random train set:")
print(train_set.columns.tolist())

print("\nColumns in random test set:")
print(test_set.columns.tolist())

income_cat removed from housing, train_set, and test_set (if present).

Columns in housing:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

Columns in random train set:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

Columns in random test set:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']


## Section Summary — Create a Test Set

In this section, we completed the test-set creation workflow from Chapter 2.

### What we did
1. Loaded the California housing dataset
2. Inspected its structure
3. Created a simple random train/test split
4. Created `income_cat` from `median_income`
5. Used stratified sampling to preserve income-category proportions
6. Compared random and stratified test-set distributions
7. Removed the temporary helper column

### Key learning
A pure random split is simple, but it may not preserve important subgroup proportions.

A stratified split is often better when:
- a variable is especially important
- you want train and test sets to reflect the full dataset more accurately

In this project, stratified sampling based on `median_income` produced a more representative test set.

## Stopping Point

This notebook has completed the Chapter 2 section **Create a Test Set**.

Next step in the book:
**Explore and Visualize the Data to Gain Insights**